# Exploratory Data Analysis (EDA) — Telco Customer Churn

## What is EDA?
Before building any model, we need to **understand our data**. EDA is the process of:
1. **Looking at the raw data** — what columns do we have? What do the values look like?
2. **Checking data quality** — are there missing values? Outliers? Wrong data types?
3. **Understanding distributions** — what's the range of values? Are they balanced?
4. **Finding patterns** — which features seem related to churn? This guides feature selection.

EDA is like a doctor examining a patient before prescribing treatment. You wouldn't prescribe without examining first.

## How to use this notebook
Run each cell top-to-bottom with `Shift+Enter`. The outputs (tables, charts) appear below each cell.

**Prerequisite:** Place the Kaggle CSV at `../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`

In [ ]:
# --- Setup: Import Libraries ---
# pandas: data manipulation (DataFrames)
# numpy: numerical operations
# matplotlib/seaborn: data visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set plot style — 'whitegrid' adds subtle grid lines, easier to read values
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully!')

In [ ]:
# --- Load the Dataset ---
# pd.read_csv() reads a CSV file into a DataFrame.
# A DataFrame is like a spreadsheet in Python — rows and columns.

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nFirst 5 rows:')
df.head()

## 1. Basic Info — Data Types and Memory Usage

`df.info()` shows:
- Column names and their data types (`int64`, `float64`, `object`)
- How many non-null values exist (nulls = missing data)
- Memory usage

`object` dtype usually means a text/categorical column.

In [ ]:
df.info()

## 2. Target Variable — Churn Distribution

**Question:** How many customers churned vs. stayed?

This tells us if we have a **class imbalance** problem. If 95% of customers stayed and 5% churned, a model that always predicts "stay" would have 95% accuracy — but would be completely useless. We need to know the balance to choose the right strategy.

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('Churn Distribution:')
print(f"  No  (Stayed): {churn_counts['No']:,}  ({churn_pct['No']:.1f}%)")
print(f"  Yes (Churned): {churn_counts['Yes']:,}  ({churn_pct['Yes']:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Stayed', 'Churned'], churn_counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Customer Churn Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Stayed', 'Churned'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nInsight: ~26.5% churn rate — mild class imbalance.')
print('Strategy: Use class_weight="balanced" in Random Forest and scale_pos_weight in XGBoost.')

## 3. Data Quality Check — Missing Values

Missing values can crash your model or cause silent errors. We need to find and handle them before training.

In [ ]:
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values detected!')

print('\n--- Checking TotalCharges for hidden blanks ---')
# TotalCharges looks numeric but contains spaces for new customers
# who haven't been billed yet
blank_charges = df[df['TotalCharges'].str.strip() == '']['TotalCharges']
print(f'Blank TotalCharges (not NaN, just spaces): {len(blank_charges)} rows')
print('These are brand-new customers — we will replace blanks with 0.')

## 4. Numeric Features — Distribution Analysis

We look at how numeric features are distributed and whether they differ between churners and non-churners.

In [ ]:
# Fix TotalCharges for plotting
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip().replace('', '0'), errors='coerce').fillna(0)

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for idx, col in enumerate(numeric_cols):
    # Left: distribution by churn status
    for label, color in [('No', '#2ecc71'), ('Yes', '#e74c3c')]:
        subset = df[df['Churn'] == label][col]
        axes[idx, 0].hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={label}')
    axes[idx, 0].set_title(f'{col} Distribution by Churn', fontweight='bold')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Count')
    axes[idx, 0].legend()

    # Right: box plot
    churned = df[df['Churn'] == 'Yes'][col]
    stayed = df[df['Churn'] == 'No'][col]
    axes[idx, 1].boxplot([stayed, churned], labels=['Stayed', 'Churned'])
    axes[idx, 1].set_title(f'{col} — Stayed vs Churned', fontweight='bold')
    axes[idx, 1].set_ylabel(col)

plt.suptitle('Numeric Features: Churn vs No Churn', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary stats
print('\nMean values by Churn status:')
print(df.groupby('Churn')[numeric_cols].mean().round(2))

**Key Insights from Numeric Features:**
- **Tenure:** Churned customers have much lower tenure → new customers churn more
- **MonthlyCharges:** Churned customers pay MORE monthly on average
- **TotalCharges:** Churned customers have lower totals (because they leave sooner)

## 5. Categorical Features — Churn Rate by Category

For each categorical column, we plot the churn rate per category. This reveals which categories are high-risk.

In [ ]:
categorical_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']

fig, axes = plt.subplots(len(categorical_cols), 1, figsize=(12, 5 * len(categorical_cols)))

for idx, col in enumerate(categorical_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
    churn_rate.columns = [col, 'Churn Rate (%)']
    churn_rate = churn_rate.sort_values('Churn Rate (%)', ascending=False)

    bars = axes[idx].barh(churn_rate[col], churn_rate['Churn Rate (%)'],
                          color=['#e74c3c' if x > 30 else '#3498db' for x in churn_rate['Churn Rate (%)']])
    axes[idx].set_title(f'Churn Rate by {col}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Churn Rate (%)')
    axes[idx].axvline(x=26.5, color='gray', linestyle='--', alpha=0.7, label='Overall avg (26.5%)')
    axes[idx].legend()

    for bar, val in zip(bars, churn_rate['Churn Rate (%)']):
        axes[idx].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                       f'{val:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Correlation Heatmap — Numeric Features

A correlation matrix shows how strongly each pair of numeric features is related.
- **+1.0** = perfect positive correlation (when one goes up, so does the other)
- **-1.0** = perfect negative correlation (when one goes up, the other goes down)
- **0.0** = no relationship

High correlations between features (multicollinearity) can sometimes hurt model performance.

In [ ]:
# Create binary churn column for correlation analysis
df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_binary']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle (it's symmetric)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with Churn:')
print(corr_matrix['Churn_binary'].drop('Churn_binary').sort_values(key=abs, ascending=False))

## 7. Summary — Key Findings

This section summarizes what the EDA taught us, which directly informed the modeling choices in `train.py`.

In [ ]:
print("""
=== EDA SUMMARY ===

1. CLASS IMBALANCE
   - 73.5% No-Churn vs 26.5% Churn
   - Mild imbalance — handle with class_weight='balanced'
   - Don't use Accuracy as the primary metric; use F1 + ROC-AUC

2. DATA QUALITY
   - TotalCharges has 11 blank entries (not NaN) → replace with 0
   - No other missing values
   - customerID is just an ID — drop it

3. STRONGEST CHURN PREDICTORS (from visual analysis)
   - Contract type: Month-to-month → 43% churn vs Two-year → 3%
   - Tenure: churners average ~18 months vs stayers ~37 months
   - Internet: Fiber optic → 42% churn vs DSL → 19%
   - OnlineSecurity: No → 42% churn vs Yes → 15%
   - TechSupport: No → 42% churn vs Yes → 15%
   - PaymentMethod: Electronic check → 45% churn (highest risk)

4. MODELING IMPLICATIONS
   - Scale numeric features (different ranges: tenure 0-72, charges 20-120)
   - Encode categorical features (LabelEncoder for multi-class)
   - Tree-based models (RF, XGBoost) likely to perform well given mix
     of numeric and categorical features
   - Feature importance will likely rank Contract and tenure at top
""")